# Run 1 (Baseline Replication) with Inference Loop Fix

This notebook trains the **Run 1 baseline configuration** and synthesizes Marathi speech using the **corrected inference loop**:
- **Configuration**: 1,200 samples, 3 epochs (450 steps), Attention-only LoRA (`q_proj`, `k_proj`, `v_proj`, `o_proj`, 9.18M parameters), lr=1e-4, warmup=10.
- **Inference Fix Applied**: `adaptive_max = min(2520, max(280, len(text) * 14))` and `repetition_penalty = 1.1` to eliminate the 30.7s infinite silence loop on Sentence 01.
- **Hardware**: Kaggle NVIDIA Tesla T4 GPU (~1h 40m total runtime).

In [ ]:
# 1. Clone repository and install dependencies
import os
repo_dir = "/kaggle/working/indic-speak-marathi-finetune"
if not os.path.exists(repo_dir):
    !git clone https://github.com/mehersoni/indic-speak-marathi-finetune.git {repo_dir}
%cd {repo_dir}
!git pull

# Remove conflicting pre-installed torchao on Kaggle and install dependencies
!pip uninstall -y torchao
!pip install -q "transformers>=5" peft accelerate snac soundfile datasets pandas pyarrow huggingface_hub pyyaml

In [ ]:
# 2. Hugging Face Authentication
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "")

if hf_token:
    login(token=hf_token)
    os.environ["HF_TOKEN"] = hf_token
    print("Hugging Face authenticated successfully.")
else:
    print("HF_TOKEN not detected in Kaggle Secrets. If needed, set os.environ['HF_TOKEN'] = 'your_token'.")

In [ ]:
# 3. Prepare and filter Marathi dataset
!CUDA_VISIBLE_DEVICES=0 python scripts/prepare_dataset.py

In [ ]:
# 4. Run GPU Smoke Test with Run 1 Config (Attention-only LoRA, batch=2, fp16)
!CUDA_VISIBLE_DEVICES=0 python scripts/smoke_test.py --config configs/marathi_lora_run1.yaml

In [ ]:
# 5. Train Run 1 Baseline LoRA adapter (1,200 samples, 450 steps, ~1h 40m)
!CUDA_VISIBLE_DEVICES=0 python src/train.py --config configs/marathi_lora_run1.yaml

In [ ]:
# 6. Generate speech samples with Base Model (for baseline comparison)
!CUDA_VISIBLE_DEVICES=0 python src/inference.py \
    --model bodhan-ai/indic-speak \
    --out-dir outputs/examples_run1

In [ ]:
# 7. Generate speech samples with Run 1 Fine-Tuned Model (WITH LOOP FIX)
# Note: src/inference.py now automatically applies adaptive_max = min(2520, max(280, len(text)*14))
# and repetition_penalty=1.1, eliminating the 30.7s runaway loop on sentence 01.
!CUDA_VISIBLE_DEVICES=0 python src/inference.py \
    --model bodhan-ai/indic-speak \
    --adapter outputs/lora_marathi_run1/final_adapter \
    --out-dir outputs/examples_run1

In [ ]:
# 8. Audio Diagnostic & In-Notebook Audio Player
import soundfile as sf
import numpy as np
from pathlib import Path
import IPython.display as ipd

audio_dir = Path("outputs/examples_run1")
wav_files = sorted(audio_dir.glob("*.wav"))

print(f"=== Run 1 Generated Audio Diagnostics ({len(wav_files)} files) ===")
print(f"{'Filename':<22} | {'Duration':>8} | {'RMS Energy':>10} | {'Peak Amp':>8} | {'Status'}")
print("-" * 70)

for wf in wav_files:
    wav, sr = sf.read(str(wf))
    dur = len(wav) / sr
    rms = float(np.sqrt(np.mean(wav**2)))
    peak = float(np.max(np.abs(wav)))
    status = "Fixed (No Loop)" if dur < 10.0 else "Warning (Long)"
    print(f"{wf.name:<22} | {dur:>7.2f}s | {rms:>10.4f} | {peak:>8.4f} | {status}")

print("\n--- Audio Playback ---")
for wf in wav_files:
    if wf.name.startswith("finetuned"):
        print(f"▶ Playing: {wf.name}")
        ipd.display(ipd.Audio(str(wf)))

In [ ]:
# 9. Package Run 1 Deliverables for Download
import shutil

sub_dir = "/kaggle/working/run1_fixed_submission"
os.makedirs(f"{sub_dir}/audio", exist_ok=True)
os.makedirs(f"{sub_dir}/model", exist_ok=True)

for f in os.listdir("outputs/examples_run1"):
    if f.endswith(".wav"):
        shutil.copy(f"outputs/examples_run1/{f}", f"{sub_dir}/audio/{f}")

adapter_src = "outputs/lora_marathi_run1/final_adapter"
if os.path.exists(adapter_src):
    shutil.copytree(adapter_src, f"{sub_dir}/model/final_adapter", dirs_exist_ok=True)

state_file = "outputs/lora_marathi_run1/trainer_state.json"
if os.path.exists(state_file):
    shutil.copy(state_file, f"{sub_dir}/trainer_state.json")

zip_out = "/kaggle/working/run1_fixed_submission"
shutil.make_archive(zip_out, "zip", sub_dir)
print(f"\n✓ Download ready: {zip_out}.zip")
print("Download from the Kaggle Output tab on the right.")